
# Market Basket Analysis (Apriori) 
This notebook:
1. Loads a CSV of **user_id**s (first column).
2. Loads Instacart **PRIOR** data (`orders.csv`, `order_products__prior.csv`, `products.csv`) and filters to those users.
3. Builds transactions (one basket per prior order).
4. Runs **Apriori** to find frequent itemsets, then computes **support**, **confidence**, **lift**, and **interest** for association rules.
5. Includes helpers for querying rules and making basket-completion recommendations.

## Metrics refresher
Let A be an itemset (antecedent) and B a single item (consequent), N = number of baskets.
- **support(X)** = baskets containing X / N  
- **confidence(A→B)** = support(A ∪ {B}) / support(A)  
- **lift(A→B)** = confidence(A→B) / support(B) = support(A∪B)/(support(A)·support(B))  
- **interest(A→B)** = confidence(A→B) − support(B)  (additive improvement over baseline)


## 0) Setup & Paths

In [16]:

# %pip install pandas numpy mlxtend matplotlib
import os, itertools
import numpy as np
import pandas as pd

from collections import defaultdict
from typing import List

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option("display.max_colwidth", 120)

# --- EDIT THESE PATHS IF NEEDED ---
DATA_DIR = "data"          # folder with Instacart CSVs
USERS_FILE = "user_partitions.csv"  # CSV with user_id in the first column (header optional)

ORDERS_FILE = os.path.join(DATA_DIR, "orders.csv")
ORDER_LINES_FILE = os.path.join(DATA_DIR, "order_products__prior.csv")  # PRIOR for history
PRODUCTS_FILE = os.path.join(DATA_DIR, "products.csv")

print("Paths set.")


Paths set.


## 1) Load user list and PRIOR data

In [17]:
# Load user list (CSV has columns: user_id, partition)
users_df = pd.read_csv(USERS_FILE)

# Optional: filter a specific partition (uncomment and set the value you want)
# users_df = users_df[users_df["partition"] == "train"]  # e.g., "train" / "valid" / "test"

# Ensure we have a clean integer set of user_ids
if "user_id" not in users_df.columns:
    users_df = users_df.rename(columns={users_df.columns[0]: "user_id"})
user_series = pd.to_numeric(users_df["user_id"], errors="coerce").dropna().astype("int64")
USER_IDS = set(user_series.tolist())
print("Loaded user ids:", len(USER_IDS))

# Load Instacart data
orders = pd.read_csv(ORDERS_FILE, usecols=["order_id","user_id","order_number","days_since_prior_order"])
prior = pd.read_csv(ORDER_LINES_FILE, usecols=["order_id","product_id"])
products = pd.read_csv(PRODUCTS_FILE, usecols=["product_id","product_name"])

# Join product names and orders
data = (prior.merge(products, on="product_id", how="left")
              .merge(orders, on="order_id", how="left"))

# Filter to selected users
data_sel = data[data["user_id"].isin(USER_IDS)].copy()
num_orders = data_sel["order_id"].nunique()
num_items = data_sel["product_name"].nunique()
print("Selected users:", len(USER_IDS), "| prior orders kept:", num_orders, "| unique items:", num_items)

display(data_sel.head(3))


Loaded user ids: 1000
Selected users: 1000 | prior orders kept: 15633 | unique items: 15246


,order_id,product_id,product_name,user_id,order_number,days_since_prior_order
5030,532,46665,Pesto Alla Genovese Basil,100845,7,2.0
5031,532,33198,Sparkling Natural Mineral Water,100845,7,2.0
5032,532,48857,Authentic French Brioche,100845,7,2.0


## 2) Build transactions (one basket per prior order)

In [18]:

baskets_df = data_sel.groupby("order_id")["product_name"].apply(lambda x: sorted(set(x))).reset_index()
transactions = baskets_df["product_name"].tolist()
N = len(transactions)
U = len(set(i for t in transactions for i in t))
print("Baskets:", N, "| Unique items:", U)


Baskets: 15633 | Unique items: 15246


## 3) Apriori: frequent itemsets and association rules

In [19]:
# One-hot encode
te = TransactionEncoder().fit(transactions)
X = pd.DataFrame(te.transform(transactions), columns=te.columns_)

N = len(transactions)
U = X.shape[1]
print(f"[DBG] Baskets N={N}, Unique items U={U}")

# Try a ladder of supports until we get some 2-itemsets
support_grid = [0.05, 0.03, 0.02, 0.01, 0.005, max(1.0/N, 0.001)]
support_grid = sorted(set([s for s in support_grid if s > 0]), reverse=True)

freq = pd.DataFrame()
chosen_support = None
for s in support_grid:
    f = apriori(X, min_support=s, use_colnames=True, max_len=3)
    pairs = f[f["itemsets"].apply(lambda s: len(s)==2)]
    print(f"[DBG] min_support={s:.4f} -> itemsets={len(f)} (pairs={len(pairs)})")
    if len(pairs) > 0 or (len(f) > 0 and s == support_grid[-1]):
        freq, chosen_support = f.sort_values("support", ascending=False), s
        break

if freq.empty:
    raise ValueError("No frequent itemsets found. Consider lowering min_support further or widening to more users.")

print(f"[INFO] Using min_support={chosen_support:.4f}. "
      f"Singles={len(freq[freq['itemsets'].str.len()==1])}, "
      f"Pairs={len(freq[freq['itemsets'].str.len()==2])}, "
      f"Triples={len(freq[freq['itemsets'].str.len()==3])}")

# Rules (don’t prune yet)
rules = association_rules(freq, metric="confidence", min_threshold=0.1).copy()

# Interest = confidence - support(consequent)
singletons = freq[freq["itemsets"].apply(lambda s: len(s)==1)][["itemsets","support"]]
conseq_support = {list(s)[0]: sup for s, sup in zip(singletons["itemsets"], singletons["support"])}

def get_consequent_support(cs: frozenset) -> float:
    return conseq_support.get(next(iter(cs)), np.nan)

rules["interest"] = rules["confidence"] - rules["consequents"].apply(get_consequent_support)

print(f"[INFO] Raw rules: {len(rules)}")
if len(rules) == 0:
    print("[HINT] No rules produced. You need at least some 2-itemsets; try lowering min_support or checking that baskets aren’t empty.")
else:
    # Optional pruning (comment out if you still get 0 rows)
    pruned = rules[(rules["lift"] > 1.0) & (rules["support"] >= chosen_support)].copy()
    print(f"[INFO] After pruning (lift>1 & support>=min_support): {len(pruned)} rules")
    # choose which to display/save
    rules_to_show = pruned if len(pruned) > 0 else rules
    rules_to_show = rules_to_show.sort_values(["lift","confidence","support"], ascending=False)

    display(freq.head(10))
    display(rules_to_show[["antecedents","consequents","support","confidence","lift","interest"]].head(20))

    # Save outputs
    freq_out = "data/selected_users_itemsets_apriori.csv"
    rules_out = "data/selected_users_rules_apriori.csv"
    freq.to_csv(freq_out, index=False)
    rules_to_show.to_csv(rules_out, index=False)
    print("Saved itemsets to:", freq_out)
    print("Saved rules to:", rules_out)

[DBG] Baskets N=15633, Unique items U=15246
[DBG] min_support=0.0500 -> itemsets=6 (pairs=0)
[DBG] min_support=0.0300 -> itemsets=17 (pairs=0)
[DBG] min_support=0.0200 -> itemsets=43 (pairs=1)
[INFO] Using min_support=0.0200. Singles=42, Pairs=1, Triples=0
[INFO] Raw rules: 2
[INFO] After pruning (lift>1 & support>=min_support): 2 rules


,support,itemsets
4,0.124672,(Banana)
3,0.113158,(Bag of Organic Bananas)
31,0.089810,(Organic Strawberries)
15,0.079639,(Organic Baby Spinach)
25,0.064991,(Organic Hass Avocado)
12,0.055652,(Organic Avocado)
10,0.047144,(Limes)
9,0.046312,(Large Lemon)
32,0.043114,(Organic Whole Milk)
40,0.042090,(Strawberries)


,antecedents,consequents,support,confidence,lift,interest
0,(Organic Strawberries),(Bag of Organic Bananas),0.024499,0.272792,2.410717,0.159634
1,(Bag of Organic Bananas),(Organic Strawberries),0.024499,0.216507,2.410717,0.126696


Saved itemsets to: data/selected_users_itemsets_apriori.csv
Saved rules to: data/selected_users_rules_apriori.csv


## 4) Helpers — query rules and recommend items

In [ ]:

# Index rules (only single-item consequents)
rules_idx = []
for _, r in rules.iterrows():
    A = tuple(sorted(list(r["antecedents"])))
    C = list(r["consequents"])[0] if len(r["consequents"])==1 else None
    if C is None:
        continue
    rules_idx.append((A, C, r["lift"], r["confidence"], r["support"], r["interest"]))

def what_goes_with(item: str, top_k: int=10) -> pd.DataFrame:
    sub = [row for row in rules_idx if row[1] == item]
    if not sub:
        return pd.DataFrame(columns=["antecedent","consequent","support","confidence","lift","interest"])
    sub_sorted = sorted(sub, key=lambda x: (x[2], x[3], x[4]), reverse=True)[:top_k]
    return pd.DataFrame([(a,(c),s,conf,lft,intt) for a,c,lft,conf,s,intt in sub_sorted],
                        columns=["antecedent","consequent","support","confidence","lift","interest"])

def recommend_for_basket(current_items: List[str], k: int=10, subset_k: int=2) -> pd.DataFrame:
    S = sorted(set(current_items))
    best = defaultdict(lambda: 0.0)
    meta = {}
    for A, C, lift_, conf_, supp_, intr_ in rules_idx:
        if len(A) <= subset_k and set(A).issubset(S):
            score = lift_ * conf_
            if C not in S and score > best[C]:
                best[C] = score
                meta[C] = {"antecedent": A, "lift": lift_, "confidence": conf_, "support": supp_, "interest": intr_}
    rows = [(i, s, meta[i]["antecedent"], meta[i]["lift"], meta[i]["confidence"], meta[i]["support"], meta[i]["interest"]) 
            for i, s in best.items()]
    recs = pd.DataFrame(rows, columns=["item","score","because","lift","confidence","support","interest"]).sort_values("score", ascending=False).head(k)
    return recs

print(f"[DBG] rules rows: {len(rules)}")
print(f"[DBG] rules_idx entries: {len(rules_idx)}")

# Demo basket: top-3 most frequent items among selected users (fall back to 1–2 if needed)
popular = (data_sel.groupby("product_name")
                    .size()
                    .sort_values(ascending=False)
                    .head(3)
                    .index.tolist())
if not popular:
    # fallback: any items at all?
    popular = list(data_sel["product_name"].value_counts().head(3).index)
print("[DBG] Demo basket:", popular)

# Try with subset_k up to 3 in case your rules use larger antecedents
recs = recommend_for_basket(popular, k=10, subset_k=3) if popular else pd.DataFrame()

if recs is None or recs.empty:
    # Explain why nothing showed
    note_rows = []
    if len(rules_idx) == 0:
        note_rows.append("No rules available. Lower min_support/confidence or remove extra pruning.")
    if not popular:
        note_rows.append("Demo basket is empty—check data_sel or pick a manual basket like ['Milk','Bread'].")
    if not note_rows:
        note_rows.append("No rules matched this basket. Try increasing subset_k or use a different basket.")
    recs = pd.DataFrame({"note": note_rows})
    print("[INFO] No recommendations to display; see notes below.")

display(recs)


Demo basket: ['Banana', 'Bag of Organic Bananas', 'Organic Strawberries']


,item,score,because,lift,confidence,support,interest
